In [1]:
import os 
import h5py 
import numpy as np 
import matplotlib.pyplot as plt 
import pandas as pd 
from gecatsim.pyfiles.GetMu import GetMu
from matplotlib.patches import Rectangle
import sys
from src.lutpy.spr_calc.hu_to_spr import hu_to_spr
from pathlib import Path
import re
sys.path.append(r"C:\Users\Nelly Kleppe\PycharmProjects\exjobb\calibration\lutpy_main")



In [2]:
def mean_in_circle(img, cx_mm, cy_mm, r_mm, fov): 
    cx, cy, r = mm_centered_to_pixel(cx_mm, cy_mm, r_mm, img.shape, fov) 
    yy, xx = np.ogrid[:img.shape[0], :img.shape[1]] 
    mask = (xx - cx)**2 + (yy - cy)**2 <= r**2 
    return np.mean(img[mask]) 

def mm_centered_to_pixel(cx_mm, cy_mm, r_mm, img_shape, fov): 
    ny, nx = img_shape 
    px_size = fov/1024 
    # image center in pixel coordinates 
    x0 = (nx - 1) / 2 
    y0 = (ny - 1) / 2 
    # convert mm to pixel 
    x_px = x0 + cx_mm / px_size 
    y_px = y0 - cy_mm / px_size 
    r_px = r_mm / px_size 
    return int(round(x_px)), int(round(y_px)), r_px


In [14]:
mat_folder = r"C:\Users\Nelly Kleppe\PycharmProjects\exjobb\calibration\matfiles" # folder with .mat files 
save_folder = r"C:\Users\Nelly Kleppe\PycharmProjects\exjobb\calibration" 
save_file = "nelly_full_vols.xlsx" # output Excel file 
kev_values = list(range(40, 141, 1)) 
#files = ['head1.mat', 'head2.mat', 'head3.mat', 'head4.mat', 'head5.mat', 'body1.mat', 'body2.mat', 'body3.mat', 'body4.mat', 'body5.mat'] 
r=8
files = [
    "body1_262124_170423.mat",
    "body2_261924_180401.mat",
    "body3_261624_190434.mat",
    "body4_261424_200435.mat",
    "body5_261224_210410.mat",
    "head1_260224_220453.mat",
    "head2_265324_220441.mat",
    "head3_264424_230421.mat",
    "head4_263425_000435.mat",
    "head5_262525_010414.mat",
]

In [15]:
# Defining geometries, materials and ROI centers for body and head 
material_names = [None, 'soft_water_solid', 'soft_water_liquid', 'soft_brain', 'soft_breast', 'soft_adipose', 'soft_liver', 'lung_ln300', 'lung_ln450', 'bone_inner', 'bone_CB2_30pct_CaCO3', 'bone_CB2_50pct_CaCO3', 'bone_cortical' ] 

size_body = [200,150] # half-axes 
cx_body = [0, 0, 53.03, 73.333300, 53.03, 0, -53.03, -73.333300, -53.03, 125.85, 150, 125.85, -125.85, -150, -125.85] 
cy_body = [37.5, 73.333300, 53.03, 0, -53.03, -73.333300, -53.03, 0, 53.03, 71, 0, -71, -71, 0, 71] 
insert_material_body = [1, 1, 6, 1, 1, 7, 1, 5, 1, 1, 1, 2, 3, 4, 8] 
body_pos = pd.DataFrame({"cx":cx_body, "cy":cy_body, "insert_material":insert_material_body}) # separate copies 
body1 = body_pos.copy() 
body2 = body_pos.copy() 
body3 = body_pos.copy() 
body4 = body_pos.copy() 
body5 = body_pos.copy() 

# add unique middle inserts 
body1.loc[len(body1)] = [0, 0, 1] 
body2.loc[len(body2)] = [0, 0, 9] 
body3.loc[len(body3)] = [0, 0, 10] 
body4.loc[len(body4)] = [0, 0, 11] 
body5.loc[len(body5)] = [0, 0, 12] 
pos_map = { "body1.mat": body1, "body2.mat": body2, "body3.mat": body3, "body4.mat": body4, "body5.mat": body5, } 
# TODO remove .mat, just body1 or something instead

size_head = [100, 100] # half-axes 
cx_head = [0, 0, 53.03, 75, 53.03, 0, -53.03, -75, -53.03] 
cy_head = [37.5, 75, 53.03, 0, -53.03, -75, -53.03, 0, 53.03] 
insert_material_head = [1, 4, 6, 1, 2, 7, 3, 5, 8] 
head_pos = pd.DataFrame({"cx":cx_head, "cy":cy_head, "insert_material":insert_material_head}) # separate copies 
head1 = head_pos.copy() 
head2 = head_pos.copy() 
head3 = head_pos.copy() 
head4 = head_pos.copy() 
head5 = head_pos.copy() 

# add unique middle inserts 
head1.loc[len(head1)] = [0, 0, 1] 
head2.loc[len(head2)] = [0, 0, 9] 
head3.loc[len(head3)] = [0, 0, 10] 
head4.loc[len(head4)] = [0, 0, 11] 
head5.loc[len(head5)] = [0, 0, 12] 
pos_map.update({"head1.mat": head1, "head2.mat": head2, "head3.mat": head3, "head4.mat": head4, "head5.mat": head5})


In [17]:
rows = []

mu_PE = GetMu('polyethylene', kev_values)
mu_PVC = GetMu('pvc_legacy', kev_values)
mu_water = GetMu('water', kev_values)
mu_air = GetMu('air', kev_values)

for file in files:
    print("File: ", file)
    file_path = os.path.join(mat_folder, file)
    with h5py.File(file_path, 'r') as f:
        pe_vol = 10*np.array(f['Image_PE'])  # 12x1024x1024
        pvc_vol = 10*np.array(f['Image_PVC'])
    # TODO make more general with finding map
    pos = pos_map[file[0:5]+'.mat']  # find files that begin with body1... and head1...
    if str(file).startswith('head'):
        fov = 350
    elif str(file).startswith('body'):
        fov = 420
        
    n_slices = pe_vol.shape[0]
    for s in range(n_slices):
        print("     Slice: ", s+1, " ", round(s/n_slices, 3), "%")
        pe = pe_vol[s, :, :]
        pvc = pvc_vol[s, :, :]
        # TODO dubbelkolla dimensioner och inserts, behövs inte nu vid 3d?
        """pe = np.flipud(pe)  
        pvc = np.flipud(pvc)"""
        
        for e in range(len(kev_values)):
            mu_pe = mu_PE[e]
            mu_pvc = mu_PVC[e]
            mu_w = mu_water[e]
            mu = mu_pe*pe+mu_pvc*pvc
            mu_a = mu_air[e]
            HU = 1000*(mu-mu_w)/(mu_w-mu_a)
            for i in range(len(pos)):
                cx = (pos.iloc[i]["cx"])
                cy = (pos.iloc[i]["cy"])
                material = pos.iloc[i]["insert_material"]
                val = mean_in_circle(HU, cx, cy, r, fov)
    
                rows.append({
                    "file_name": file,
                    "kev": kev_values[e],
                    "insert_material": material,
                    "mean_hu": val,
                    "slice": s
                })
print("Finished")
df = pd.DataFrame(rows)
df.head()

File:  body1_262124_170423.mat
     Slice:  1   0.0 %
     Slice:  2   0.083 %
     Slice:  3   0.167 %
     Slice:  4   0.25 %
     Slice:  5   0.333 %
     Slice:  6   0.417 %
     Slice:  7   0.5 %
     Slice:  8   0.583 %
     Slice:  9   0.667 %
     Slice:  10   0.75 %
     Slice:  11   0.833 %
     Slice:  12   0.917 %
File:  body2_261924_180401.mat
     Slice:  1   0.0 %
     Slice:  2   0.083 %
     Slice:  3   0.167 %
     Slice:  4   0.25 %
     Slice:  5   0.333 %
     Slice:  6   0.417 %
     Slice:  7   0.5 %
     Slice:  8   0.583 %
     Slice:  9   0.667 %
     Slice:  10   0.75 %
     Slice:  11   0.833 %
     Slice:  12   0.917 %
File:  body3_261624_190434.mat
     Slice:  1   0.0 %
     Slice:  2   0.083 %
     Slice:  3   0.167 %
     Slice:  4   0.25 %
     Slice:  5   0.333 %
     Slice:  6   0.417 %
     Slice:  7   0.5 %
     Slice:  8   0.583 %
     Slice:  9   0.667 %
     Slice:  10   0.75 %
     Slice:  11   0.833 %
     Slice:  12   0.917 %
File:  body4_261

,file_name,kev,insert_material,mean_hu,slice
0,body1_262124_170423.mat,40,1.0,0.341417,0
1,body1_262124_170423.mat,40,1.0,0.104270,0
2,body1_262124_170423.mat,40,6.0,76.875268,0
3,body1_262124_170423.mat,40,1.0,-0.326467,0
4,body1_262124_170423.mat,40,1.0,-0.156912,0


In [18]:
# average for each material and energy

df = df.drop(columns=["file_name", "slice"])
output = (
    df
    .groupby(["kev", "insert_material"], as_index=False)["mean_hu"]
    .mean()
)

# map index → name
material_map = dict(enumerate(material_names))
output["material_name"] = output["insert_material"].astype(int).map(material_map)

# remove prefix
output["insert"] = output["material_name"].str.replace(
    "Schneider_gammex_", "", regex=False
)

# drop old column if you don’t want it
output = output.drop(columns=["insert_material"])
output = output[["kev", "insert", "mean_hu"]]

output.head()

,kev,insert,mean_hu
0,40,soft_water_solid,-0.276721
1,40,soft_water_liquid,-1.852587
2,40,soft_brain,48.620808
3,40,soft_breast,-89.469022
4,40,soft_adipose,-136.418233


In [19]:
# export to excel 
output.to_excel(os.path.join(save_folder, save_file), index=False)

In [20]:
# print roi values from all phantoms
kev_check = 74
BREAST_ID = 4  # 'soft_breast'

print(f"\n=== BREAST ROI VALUES AT {kev_check} keV ===")

for file in files:
    file_path = os.path.join(mat_folder, file)

    with h5py.File(file_path, 'r') as f:
        pe = 10 * np.array(f["Image_PE"])
        pvc = 10 * np.array(f["Image_PVC"])

    pe = np.flipud(pe)
    pvc = np.flipud(pvc)

    pos = pos_map[file]
    fov = 350 if file.startswith("head") else 420

    # get index for 74 keV
    e_idx = kev_values.index(kev_check)

    mu_pe = mu_PE[e_idx]
    mu_pvc = mu_PVC[e_idx]
    mu_w = mu_water[e_idx]
    mu_a = mu_air[e_idx]

    mu = mu_pe * pe + mu_pvc * pvc
    HU = 1000 * (mu - mu_w) / (mu_w - mu_a)

    for i in range(len(pos)):
        material_id = int(pos.iloc[i]["insert_material"])

        if material_id != BREAST_ID:
            continue

        cx = pos.iloc[i]["cx"]
        cy = pos.iloc[i]["cy"]
        r = 8
        val = mean_in_circle(HU, cx, cy, r, fov)

        print(
            f"{file:10s} | ROI idx {i:2d} | (cx,cy)=({cx:6.2f},{cy:6.2f}) | HU={val:10.3f}"
        )


=== BREAST ROI VALUES AT 74 keV ===


KeyError: 'body1_262124_170423.mat'

In [ ]:
kev_check = 74

print(f"\n=== ALL HEAD INSERT ROI VALUES AT {kev_check} keV ===")

for file in [f for f in files if f.startswith("head")]:
    file_path = os.path.join(mat_folder, file)

    with h5py.File(file_path, 'r') as f:
        pe = 10 * np.array(f["Image_PE"])
        pvc = 10 * np.array(f["Image_PVC"])

    pe = np.flipud(pe)
    pvc = np.flipud(pvc)

    pos = pos_map[file]
    fov = 350

    e_idx = kev_values.index(kev_check)
    mu_pe = mu_PE[e_idx]
    mu_pvc = mu_PVC[e_idx]
    mu_w = mu_water[e_idx]
    mu_a = mu_air[e_idx]

    mu = mu_pe * pe + mu_pvc * pvc
    HU = 1000 * (mu - mu_w) / (mu_w - mu_a)

    print(f"\n--- {file} ---")
    for i in range(len(pos)):
        cx = pos.iloc[i]["cx"]
        cy = pos.iloc[i]["cy"]
        material = int(pos.iloc[i]["insert_material"])
        val = mean_in_circle(HU, cx, cy, r, fov)

        print(
            f"ROI idx {i:2d} | material {material:2d} | "
            f"(cx,cy)=({cx:7.2f},{cy:7.2f}) | HU={val:9.3f}"
        )

In [ ]:
# plot rois
from matplotlib.patches import Circle

kev_check = 74
file = "head1.mat"

file_path = os.path.join(mat_folder, file)
with h5py.File(file_path, 'r') as f:
    pe = 10 * np.array(f["Image_PE"])
    pvc = 10 * np.array(f["Image_PVC"])

pe = np.flipud(pe)
pvc = np.flipud(pvc)

e_idx = kev_values.index(kev_check)
mu_pe = mu_PE[e_idx]
mu_pvc = mu_PVC[e_idx]
mu_w = mu_water[e_idx]
mu_a = mu_air[e_idx]

mu = mu_pe * pe + mu_pvc * pvc
HU = 1000 * (mu - mu_w) / (mu_w - mu_a)

pos = pos_map[file]
fov = 350

fig, ax = plt.subplots(figsize=(8, 8))
im = ax.imshow(HU, cmap="gray_r")
plt.colorbar(im, ax=ax, label="HU")
ax.set_title(f"{file} at {kev_check} keV")
ax.axis("off")

for i in range(len(pos)):
    cx = pos.iloc[i]["cx"]
    cy = pos.iloc[i]["cy"]
    material = int(pos.iloc[i]["insert_material"])

    x_px, y_px, r_px = mm_centered_to_pixel(cx, cy, r, HU.shape, fov)
    val = mean_in_circle(HU, cx, cy, r, fov)

    ax.add_patch(Circle((x_px, y_px), r_px, edgecolor="red", facecolor="none", linewidth=2))
    ax.text(x_px + 10, y_px, f"{i}:{material}\n{val:.1f}", color="yellow", fontsize=9)

plt.show()

In [ ]:
# print all pixel values in roi
kev_check = 74
file = "head1.mat"
#BREAST_ID = 4  # breast
BREAST_ID = 4 # adipose 

file_path = os.path.join(mat_folder, file)

with h5py.File(file_path, 'r') as f:
    pe = 10 * np.array(f["Image_PE"])
    pvc = 10 * np.array(f["Image_PVC"])

pe = np.flipud(pe)
pvc = np.flipud(pvc)

# get correct position table
pos = pos_map[file]
fov = 350  # head

# get energy index
e_idx = kev_values.index(kev_check)

mu_pe = mu_PE[e_idx]
mu_pvc = mu_PVC[e_idx]
mu_w = mu_water[e_idx]
mu_a = mu_air[e_idx]

# compute HU image
mu = mu_pe * pe + mu_pvc * pvc
HU = 1000 * (mu - mu_w) / (mu_w - mu_a)

# find breast ROI
for i in range(len(pos)):
    material_id = int(pos.iloc[i]["insert_material"])
    if material_id != BREAST_ID:
        continue

    cx = pos.iloc[i]["cx"]
    cy = pos.iloc[i]["cy"]

    # convert to pixel coords
    x_px, y_px, r_px = mm_centered_to_pixel(cx, cy, r, HU.shape, fov)

    yy, xx = np.ogrid[:HU.shape[0], :HU.shape[1]]
    mask = (xx - x_px) ** 2 + (yy - y_px) ** 2 <= r_px ** 2

    roi_pixels = HU[mask]

    print(f"\n=== ADIPOSE ROI PIXELS ({file}, {kev_check} keV) ===")
    print(f"Number of pixels: {roi_pixels.size}")
    print("Values:")
    print(roi_pixels)

    # optional quick stats
    print("\nStats:")
    print(
        f"min={roi_pixels.min():.3f}, max={roi_pixels.max():.3f}, mean={roi_pixels.mean():.3f}, std={roi_pixels.std():.3f}")

In [ ]:
kev_check = 74
SOLID_WATER_ID = 1  # 'soft_water_solid'

print(f"\n=== SOLID WATER ROI VALUES AT {kev_check} keV ===")

for file in files:
    file_path = os.path.join(mat_folder, file)

    with h5py.File(file_path, 'r') as f:
        pe = 10 * np.array(f["Image_PE"])
        pvc = 10 * np.array(f["Image_PVC"])

    pe = np.flipud(pe)
    pvc = np.flipud(pvc)

    pos = pos_map[file]
    fov = 350 if file.startswith("head") else 420

    e_idx = kev_values.index(kev_check)

    mu_pe = mu_PE[e_idx]
    mu_pvc = mu_PVC[e_idx]
    mu_w = mu_water[e_idx]
    mu_a = mu_air[e_idx]

    mu = mu_pe * pe + mu_pvc * pvc
    HU = 1000 * (mu - mu_w) / (mu_w - mu_a)

    print(f"\n--- {file} ---")

    found = False
    for i in range(len(pos)):
        material_id = int(pos.iloc[i]["insert_material"])

        if material_id != SOLID_WATER_ID:
            continue

        found = True
        cx = pos.iloc[i]["cx"]
        cy = pos.iloc[i]["cy"]

        val = mean_in_circle(HU, cx, cy, r, fov)

        print(
            f"ROI idx {i:2d} | (cx,cy)=({cx:8.3f},{cy:8.3f}) | HU={val:10.3f}"
        )

    if not found:
        print("No solid water inserts found")

# SPR

In [ ]:
def VMI(pe, pvc, E):
    mu_PE = np.array(GetMu('polyethylene', E))
    mu_PVC = np.array(GetMu('pvc_legacy', E))
    mu_water = np.array(GetMu('water', E))
    mu_air = np.array(GetMu('air', E))
    
    mu = mu_PE*pe+mu_PVC*pvc
    HU = 1000*(mu-mu_water)/(mu_water-mu_air)
    return HU

In [ ]:
# optimal energy pairs [keV]
E_low = 41
E_high = 45
phantom_file = 'head1.mat'
md = True # True: ska använda md-filer, inte mono, False: två färdiga vmier

if md:
    file_path = os.path.join(mat_folder, phantom_file)
    with h5py.File(file_path, 'r') as f:
        pe = 10*np.array(f['Image_PE'])
        pvc = 10*np.array(f['Image_PVC'])
    pe = np.flipud(pe)
    pvc = np.flipud(pvc)
    VMI_low = VMI(pe, pvc, E_low)
    VMI_high = VMI(pe, pvc, E_high)
   
# TODO 
if not md:
    # enter filenames
    VMI_low = ...
    VMI_high = ...


mu_w_low = GetMu('water', E_low)[0]
mu_w_high = GetMu('water', E_high)[0]


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

images = [
    (pe, f"PE"),
    (pvc, f"PVC"),
    (VMI_low, f"VMI low ({E_low} keV)"),
    (VMI_high, f"VMI high ({E_high} keV)"),
    (mu_low, f"mu low ({E_low} keV)"),
    (mu_high, f"mu high ({E_high} keV)"),
]

for ax, (img, title) in zip(axes.ravel(), images):
    im = ax.imshow(img, cmap="gray")
    ax.set_title(title)
    ax.axis("off")
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

mid_y, mid_x = VMI_low.shape[0] // 2, VMI_low.shape[1] // 2

print("Shapes:")
print("PE:", pe.shape, pe.dtype)
print("PVC:", pvc.shape, pvc.dtype)
print("VMI_low:", VMI_low.shape, VMI_low.dtype)
print("VMI_high:", VMI_high.shape, VMI_high.dtype)
print("mu_low:", mu_low.shape, mu_low.dtype)
print("mu_high:", mu_high.shape, mu_high.dtype)

print("\nMin / max:")
print("PE:", np.min(pe), np.max(pe))
print("PVC:", np.min(pvc), np.max(pvc))
print("VMI_low:", np.min(VMI_low), np.max(VMI_low))
print("VMI_high:", np.min(VMI_high), np.max(VMI_high))
print("mu_low:", np.min(mu_low), np.max(mu_low))
print("mu_high:", np.min(mu_high), np.max(mu_high))

print("\nCenter pixel:")
print("PE center:", pe[mid_y, mid_x])
print("PVC center:", pvc[mid_y, mid_x])
print(f"VMI {E_low} center:", VMI_low[mid_y, mid_x])
print(f"VMI {E_high} center:", VMI_high[mid_y, mid_x])
print(f"mu {E_low} center:", mu_low[mid_y, mid_x])
print(f"mu {E_high} center:", mu_high[mid_y, mid_x])

# Plotta vmier

In [ ]:
# plot a VMI  
E = 70
mu_pe = GetMu('polyethylene', E) 
mu_pvc = GetMu('pvc_legacy', E) 
mu_w = GetMu('water', E) 
file_path = r'C:\Users\Nelly Kleppe\PycharmProjects\exjobb\calibration\matfiles\body2.mat' 
with h5py.File(file_path, 'r') as f: 
    pe = 10*np.array(f['Image_PE']) 
    pvc = 10*np.array(f['Image_PVC']) 
    pe = np.flipud(pe)
    pvc = np.flipud(pvc)
    mu = mu_pe*pe+mu_pvc*pvc 
    HU = 1000*(mu-mu_w)/mu_w 

plt.figure(figsize=(6, 6)) 
plt.imshow(HU, cmap='gray_r') 
plt.colorbar(label='HU') 
plt.title(f'HU at {E} keV') 
plt.axis('off') 

plt.show()

In [ ]:
# stats
fov = 420

rad = 5
mid = int(1024 / 2)

print("Övre kanten", np.mean(HU[20-rad:20+rad, mid-rad:mid+rad]))
print("Mitten", np.mean(HU[mid-rad:mid+rad, mid-rad:mid+rad]))

#print(np.log(np.var(HU[0:rad, 0:rad])))
#print(np.log(np.var(HU[mid-rad:mid+rad, mid-rad:mid+rad])))

cx_mm, cy_mm = 53.03, 53.03
r_mm=rad

# circular ROI
mean_circle = mean_in_circle(HU, cx_mm, cy_mm, r_mm, fov)

# square ROI
x_px, y_px, r_px = mm_centered_to_pixel(cx_mm, cy_mm, r_mm, HU.shape, fov)

x = int(round(x_px))
y = int(round(y_px))
rad = int(round(r_px))

roi_square = HU[y-rad:y+rad, x-rad:x+rad]
mean_square = np.mean(roi_square)

print("Circle mean HU:", mean_circle)
print("Square mean HU:", mean_square)

In [ ]:
from matplotlib.patches import Rectangle, Circle

fig, ax = plt.subplots(figsize=(8, 8))
im = ax.imshow(HU, cmap='gray_r')
plt.colorbar(im, ax=ax, label='HU')
ax.axis('off')

# --- top ROI ("Övre kanten")
ax.add_patch(Rectangle(
    (mid - rad, 5),        # x, y
    2 * rad,               # width
    rad,                   # height
    edgecolor='red',
    facecolor='none',
    linewidth=2
))

# --- center ROI ("Mitten")
ax.add_patch(Rectangle(
    (mid - rad, mid - rad),
    2 * rad,
    2 * rad,
    edgecolor='blue',
    facecolor='none',
    linewidth=2
))

# --- insert circular ROI
ax.add_patch(Circle(
    (x_px, y_px),
    r_px,
    edgecolor='lime',
    facecolor='none',
    linewidth=2
))

# --- insert square ROI (uses converted pixel radius)
ax.add_patch(Rectangle(
    (x - rad, y - rad),
    2 * rad,
    2 * rad,
    edgecolor='yellow',
    facecolor='none',
    linewidth=2
))

plt.show()